# Finviz Weekly - Basic Usage Guide

This notebook demonstrates basic usage of the finviz-weekly stock screening system with enhanced data sources.

## 1. Setup and Installation

```bash
# Install the package
pip install -e .

# Or with development dependencies
pip install -e .[dev]
```

In [ ]:
import pandas as pd
import sys
from pathlib import Path

# Add src to path if needed
sys.path.insert(0, str(Path.cwd().parent / "src"))

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 50)

## 2. Loading Screening Results

After running the screening pipeline, load and explore the results.

In [ ]:
# Load scored data
scored_df = pd.read_parquet("../data/latest/finviz_scored.parquet")

print(f"Loaded {len(scored_df)} stocks")
print(f"Columns: {len(scored_df.columns)}")

scored_df.head()

## 3. Exploring Enhanced Data

Check which enhanced data sources are available.

In [ ]:
# Check for enhanced columns
has_insider = "insider_net_value" in scored_df.columns
has_earnings = "earnings_avg_alpha" in scored_df.columns
has_financials = "net_margin" in scored_df.columns and "roe" in scored_df.columns

print(f"Insider data available: {has_insider}")
print(f"Earnings data available: {has_earnings}")
print(f"Financial data available: {has_financials}")

# Show enhanced columns
enhanced_cols = [col for col in scored_df.columns if col.startswith(("insider_", "earnings_", "net_margin", "roe"))]
print(f"\nEnhanced columns ({len(enhanced_cols)}):")
for col in enhanced_cols:
    print(f"  - {col}")

## 4. Insider Trading Analysis

In [ ]:
if has_insider:
    # Filter to stocks with insider activity
    insider_df = scored_df[
        scored_df["insider_net_value"].notna() & 
        (scored_df["insider_net_value"] != 0)
    ].copy()
    
    print(f"Stocks with insider activity: {len(insider_df)}")
    
    # Top insider buying
    print("\nTop 10 Insider Buying:")
    top_buying = insider_df.nlargest(10, "insider_net_value")[
        ["ticker", "company", "insider_net_value", "insider_total_buys", "insider_total_sells"]
    ]
    display(top_buying)
    
    # Insider buying/selling split
    net_buyers = (insider_df["insider_net_value"] > 0).sum()
    net_sellers = (insider_df["insider_net_value"] < 0).sum()
    print(f"\nNet buyers: {net_buyers}, Net sellers: {net_sellers}")
else:
    print("Insider data not available")

## 5. Earnings Quality Analysis

In [ ]:
if has_earnings:
    # Filter to stocks with sufficient earnings history
    earnings_df = scored_df[
        scored_df["earnings_avg_alpha"].notna() &
        (scored_df["earnings_total_events"].fillna(0) >= 2)
    ].copy()
    
    print(f"Stocks with 2+ earnings events: {len(earnings_df)}")
    
    # Top earnings beaters
    print("\nTop 10 Earnings Beaters:")
    top_earnings = earnings_df.nlargest(10, "earnings_avg_alpha")[
        ["ticker", "company", "earnings_avg_alpha", "earnings_win_rate", "earnings_total_events"]
    ]
    top_earnings["earnings_avg_alpha"] = (top_earnings["earnings_avg_alpha"] * 100).round(2)
    top_earnings["earnings_win_rate"] = (top_earnings["earnings_win_rate"] * 100).round(0)
    display(top_earnings)
    
    # Stats
    positive_alpha = (earnings_df["earnings_avg_alpha"] > 0).sum()
    print(f"\nStocks with positive avg alpha: {positive_alpha} ({positive_alpha/len(earnings_df)*100:.1f}%)")
else:
    print("Earnings data not available")

## 6. Financial Health Analysis

In [ ]:
if has_financials:
    # Filter to stocks with complete financial data
    financial_df = scored_df[
        scored_df["net_margin"].notna() &
        scored_df["roe"].notna() &
        scored_df["current_ratio"].notna()
    ].copy()
    
    print(f"Stocks with complete financial data: {len(financial_df)}")
    
    # Calculate composite health score
    financial_df["health_score"] = (
        financial_df["net_margin"].clip(0, 0.3) / 0.3 * 33.3 +
        financial_df["roe"].clip(0, 0.3) / 0.3 * 33.3 +
        financial_df["current_ratio"].clip(0, 3) / 3 * 33.3
    )
    
    # Top financial health
    print("\nTop 10 Financial Health:")
    top_health = financial_df.nlargest(10, "health_score")[
        ["ticker", "company", "net_margin", "roe", "current_ratio", "health_score"]
    ]
    top_health["net_margin"] = (top_health["net_margin"] * 100).round(1)
    top_health["roe"] = (top_health["roe"] * 100).round(1)
    top_health["health_score"] = top_health["health_score"].round(1)
    display(top_health)
    
    # Stats
    high_margin = (financial_df["net_margin"] > 0.15).sum()
    high_roe = (financial_df["roe"] > 0.15).sum()
    print(f"\nStocks with >15% net margin: {high_margin}")
    print(f"Stocks with >15% ROE: {high_roe}")
else:
    print("Financial data not available")

## 7. Enhanced Scoring Analysis

In [ ]:
# Check for enhanced scores
enhanced_scores = [col for col in scored_df.columns if col.startswith("score_") and "enhanced" in col or "insider" in col or "earnings_surprise" in col]

if enhanced_scores:
    print(f"Enhanced score columns available: {len(enhanced_scores)}")
    for score in enhanced_scores:
        print(f"  - {score}")
    
    # Compare traditional vs enhanced master scores
    if "score_master" in scored_df.columns and "score_enhanced_master" in scored_df.columns:
        print("\nTop 10 by Traditional Master Score:")
        top_trad = scored_df.nlargest(10, "score_master")[
            ["ticker", "company", "score_master"]
        ]
        display(top_trad)
        
        print("\nTop 10 by Enhanced Master Score:")
        top_enh = scored_df.nlargest(10, "score_enhanced_master")[
            ["ticker", "company", "score_enhanced_master"]
        ]
        display(top_enh)
else:
    print("Enhanced scores not available")

## 8. Screening Results - Top Themes

In [ ]:
# Load top50 files
from pathlib import Path

data_dir = Path("../data/latest")
top_files = list(data_dir.glob("top50_*.csv"))

print(f"Available themes: {len(top_files)}")
for f in top_files:
    theme_name = f.stem.replace("top50_", "")
    print(f"  - {theme_name}")

In [ ]:
# Load and display a specific theme
theme = "quality_value"  # Change this to any theme name

theme_path = data_dir / f"top50_{theme}.csv"
if theme_path.exists():
    theme_df = pd.read_csv(theme_path)
    print(f"\nTop 20 stocks in '{theme}' theme:")
    display(theme_df.head(20))
else:
    print(f"Theme '{theme}' not found")

## 9. High Conviction Stocks

Stocks appearing in multiple theme families.

In [ ]:
# Load conviction lists
conviction_path = data_dir / "conviction_2plus.csv"

if conviction_path.exists():
    conviction_df = pd.read_csv(conviction_path)
    print(f"Stocks in 2+ theme families: {len(conviction_df)}")
    
    # Top conviction
    print("\nTop 20 High Conviction Stocks:")
    display(conviction_df.head(20))
    
    # Stocks in 3+ families
    conv3 = conviction_df[conviction_df["count_families"] >= 3]
    print(f"\nStocks in 3+ families: {len(conv3)}")
    if len(conv3) > 0:
        display(conv3.head(15))
else:
    print("Conviction data not found")

## 10. Custom Analysis Examples

In [ ]:
# Example: Find stocks with insider buying + high quality score
if has_insider and "score_quality" in scored_df.columns:
    quality_insider = scored_df[
        (scored_df["insider_net_value"] > 1000000) &  # $1M+ net buying
        (scored_df["score_quality"] > 70)  # High quality score
    ].copy()
    
    print(f"\nStocks with $1M+ insider buying AND quality score >70: {len(quality_insider)}")
    if len(quality_insider) > 0:
        result = quality_insider[
            ["ticker", "company", "sector", "insider_net_value", "score_quality", "score_master"]
        ].sort_values("score_master", ascending=False)
        display(result.head(15))
else:
    print("Required data not available for this analysis")

In [ ]:
# Example: Find high-margin stocks with consistent earnings beats
if has_earnings and has_financials:
    margin_earnings = scored_df[
        (scored_df["net_margin"] > 0.20) &  # 20%+ net margin
        (scored_df["earnings_avg_alpha"] > 0.02) &  # 2%+ avg alpha
        (scored_df["earnings_total_events"] >= 3)  # 3+ earnings events
    ].copy()
    
    print(f"\nHigh-margin stocks with consistent earnings beats: {len(margin_earnings)}")
    if len(margin_earnings) > 0:
        result = margin_earnings[
            ["ticker", "company", "sector", "net_margin", "earnings_avg_alpha", "earnings_win_rate"]
        ].sort_values("net_margin", ascending=False)
        result["net_margin"] = (result["net_margin"] * 100).round(1)
        result["earnings_avg_alpha"] = (result["earnings_avg_alpha"] * 100).round(2)
        result["earnings_win_rate"] = (result["earnings_win_rate"] * 100).round(0)
        display(result.head(15))
else:
    print("Required data not available for this analysis")

## Next Steps

1. **Explore more themes**: Try different screening themes
2. **Backtest strategies**: Use the backtesting framework to validate strategies
3. **Create custom screens**: Combine multiple factors for your own screening logic
4. **Build portfolio**: Use conviction stocks as a starting point for deeper research

See the documentation for more advanced usage:
- `docs/ENHANCED_SCREENING.md` - Enhanced scoring strategies
- `docs/BACKTESTING.md` - Backtesting guide
- `README.md` - Complete feature reference